# Audit Congress Trading Signal — Agrégateurs tiers (Quiver, Capitol Trades)

Pendant logique de `house_ptr_coverage_audit.ipynb`. On **caractérise et on valide** les agrégateurs
comme complément/fallback aux sources primaires — on ne télécharge rien en masse.

**Usage strictement interne** (clause EIGA : pas d'usage commercial de ces données).

> La notebook tourne **sans clé** (mode `OFFLINE` sur un échantillon de démonstration **synthétique**,
> clairement étiqueté, jamais présenté comme réel) **et avec clé** (`QUIVER_API_KEY`).

## 0. Configuration

In [1]:
import io, os, json, time, re, logging
from pathlib import Path
from datetime import datetime
import urllib.request, urllib.error
import pandas as pd
from IPython.display import display

# --- Paramètres ---
OUT_DIR     = Path("out_aggregators_audit")
SAMPLE_DIR  = OUT_DIR / "samples"           # cache API + fixtures OFFLINE
OUT_DIR.mkdir(exist_ok=True); SAMPLE_DIR.mkdir(parents=True, exist_ok=True)

OVERLAP_YEAR = 2024                          # annee de recouvrement (>= 2016) pour la validation croisee
RATE_LIMIT_S = 0.6                           # delai entre requetes API (politesse)

QUIVER_API_KEY = os.environ.get("QUIVER_API_KEY", "").strip()
# Endpoint a verifier dans la doc Quiver (les chemins evoluent) :
#   bulk : https://api.quiverquant.com/beta/bulk/congresstrading
#   live : https://api.quiverquant.com/beta/live/congresstrading
QUIVER_URL = "https://api.quiverquant.com/beta/bulk/congresstrading"

# Sans cle -> mode OFFLINE : on lit un echantillon LOCAL (jamais l'API).
OFFLINE = (QUIVER_API_KEY == "")

# Extraction officielle House (sortie de TON pipeline Jours 1-2) pour OVERLAP_YEAR.
HOUSE_EXTRACT_PATH = OUT_DIR / f"house_official_{OVERLAP_YEAR}.csv"

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s")
log = logging.getLogger("agg_audit")

print(f"MODE : {'OFFLINE (echantillon local de demonstration)' if OFFLINE else 'LIVE (API Quiver)'}")
print(f"OVERLAP_YEAR = {OVERLAP_YEAR}  |  sorties -> {OUT_DIR}/")

MODE : OFFLINE (echantillon local de demonstration)
OVERLAP_YEAR = 2024  |  sorties -> out_aggregators_audit/


## 1. Récupération Quiver (idempotent, rate-limité, tolérant aux erreurs)

In [2]:
def _atomic_write_json(path: Path, obj):
    tmp = path.with_suffix(path.suffix + ".part")
    tmp.write_text(json.dumps(obj, ensure_ascii=False, indent=2), encoding="utf-8")
    tmp.replace(path)

def _ensure_offline_fixture(path: Path):
    """Ecrit un petit echantillon SYNTHETIQUE si absent.
    /!\\ DONNEES DE DEMO, NON REELLES : elles servent uniquement a faire tourner le pipeline hors ligne."""
    if path.exists():
        return
    demo = [
        # --- House 2024 : recoupent (en partie) l'extraction officielle de la section 5 ---
        {"Representative":"Mark Alford","ReportDate":"2024-03-31","TransactionDate":"2024-03-16","Ticker":"NVDA","Transaction":"Purchase","Range":"$1,001 - $15,000","House":"Representatives","Party":"Republican","TickerType":"Stock"},
        {"Representative":"Mark Alford","ReportDate":"2024-04-02","TransactionDate":"2024-03-18","Ticker":"MSFT","Transaction":"Sale","Range":"$15,001 - $50,000","House":"Representatives","Party":"Republican","TickerType":"Stock"},
        {"Representative":"Marjorie Taylor Greene","ReportDate":"2024-05-10","TransactionDate":"2024-04-22","Ticker":"AAPL","Transaction":"Purchase","Range":"$1,001 - $15,000","House":"Representatives","Party":"Republican","TickerType":"Stock"},
        {"Representative":"Josh Gottheimer","ReportDate":"2024-06-05","TransactionDate":"2024-05-20","Ticker":"AMZN","Transaction":"Purchase","Range":"$15,001 - $50,000","House":"Representatives","Party":"Democrat","TickerType":"Stock"},
        {"Representative":"Josh Gottheimer","ReportDate":"2024-07-01","TransactionDate":"2024-06-14","Ticker":"GOOGL","Transaction":"Purchase","Range":"$1,001 - $15,000","House":"Representatives","Party":"Democrat","TickerType":"Stock"},
        # --- House 2024 : DESACCORDS volontaires pour tester la section 5 ---
        {"Representative":"Mark Alford","ReportDate":"2024-08-01","TransactionDate":"2024-07-10","Ticker":"TSLA","Transaction":"Purchase","Range":"$1,001 - $15,000","House":"Representatives","Party":"Republican","TickerType":"Stock"},
        {"Representative":"Marjorie Taylor Greene","ReportDate":"2024-09-02","TransactionDate":"2024-08-15","Ticker":"AMD","Transaction":"Purchase","Range":"$50,001 - $100,000","House":"Representatives","Party":"Republican","TickerType":"Stock"},
        {"Representative":"Daniel Meuser","ReportDate":"2024-10-01","TransactionDate":"2024-09-12","Ticker":"PLTR","Transaction":"Purchase","Range":"$1,001 - $15,000","House":"Representatives","Party":"Republican","TickerType":"Stock"},
        # --- Senate 2024 : non comparable a l'extraction House (sert a la couverture par chambre) ---
        {"Representative":"John Boozman","ReportDate":"2024-04-14","TransactionDate":"2024-03-19","Ticker":"NVDA","Transaction":"Purchase","Range":"$1,001 - $15,000","House":"Senate","Party":"Republican","TickerType":"Stock"},
        {"Representative":"Tommy Tuberville","ReportDate":"2024-05-01","TransactionDate":"2024-04-10","Ticker":"XOM","Transaction":"Purchase","Range":"$15,001 - $50,000","House":"Senate","Party":"Republican","TickerType":"Stock"},
        # --- 2023 : pour mesurer la fenetre temporelle ---
        {"Representative":"Josh Gottheimer","ReportDate":"2023-03-15","TransactionDate":"2023-02-20","Ticker":"DIS","Transaction":"Sale","Range":"$1,001 - $15,000","House":"Representatives","Party":"Democrat","TickerType":"Stock"},
        {"Representative":"Tommy Tuberville","ReportDate":"2023-08-20","TransactionDate":"2023-07-29","Ticker":"AAPL","Transaction":"Purchase","Range":"$1,001 - $15,000","House":"Senate","Party":"Republican","TickerType":"Stock"},
    ]
    _atomic_write_json(path, demo)
    log.warning("Fixture OFFLINE de demonstration ecrite : %s (DONNEES NON REELLES)", path)

def fetch_quiver():
    """Renvoie la liste brute de records Quiver. Cache disque + rate-limit + ecriture atomique.
    OFFLINE -> lit l'echantillon local ; LIVE -> appelle l'API (header Authorization: Token)."""
    cache = SAMPLE_DIR / ("quiver_OFFLINE_FIXTURE.json" if OFFLINE else "quiver_cache.json")
    if OFFLINE:
        _ensure_offline_fixture(cache)
        return json.loads(cache.read_text(encoding="utf-8"))
    if cache.exists() and cache.stat().st_size > 0:
        log.info("Quiver : cache present (%d octets)", cache.stat().st_size)
        return json.loads(cache.read_text(encoding="utf-8"))
    req = urllib.request.Request(QUIVER_URL, headers={
        "Authorization": f"Token {QUIVER_API_KEY}", "Accept": "application/json"})
    for attempt in range(3):
        try:
            time.sleep(RATE_LIMIT_S)
            with urllib.request.urlopen(req, timeout=60) as r:
                data = json.loads(r.read().decode("utf-8"))
            _atomic_write_json(cache, data)
            log.info("Quiver : %d records recuperes", len(data))
            return data
        except urllib.error.URLError as e:
            log.warning("Quiver tentative %d echouee : %s", attempt + 1, e)
            time.sleep(2 * (attempt + 1))
    raise RuntimeError("Echec de recuperation Quiver (verifie cle / endpoint / reseau).")

raw_quiver = fetch_quiver()
print(f"{len(raw_quiver)} records bruts (source : {'fixture OFFLINE' if OFFLINE else 'API Quiver'}).")
display(pd.DataFrame(raw_quiver).head())

WARNING Fixture OFFLINE de demonstration ecrite : out_aggregators_audit/samples/quiver_OFFLINE_FIXTURE.json (DONNEES NON REELLES)


12 records bruts (source : fixture OFFLINE).


,Representative,ReportDate,TransactionDate,Ticker,Transaction,Range,House,Party,TickerType
0,Mark Alford,2024-03-31,2024-03-16,NVDA,Purchase,"$1,001 - $15,000",Representatives,Republican,Stock
1,Mark Alford,2024-04-02,2024-03-18,MSFT,Sale,"$15,001 - $50,000",Representatives,Republican,Stock
2,Marjorie Taylor Greene,2024-05-10,2024-04-22,AAPL,Purchase,"$1,001 - $15,000",Representatives,Republican,Stock
3,Josh Gottheimer,2024-06-05,2024-05-20,AMZN,Purchase,"$15,001 - $50,000",Representatives,Democrat,Stock
4,Josh Gottheimer,2024-07-01,2024-06-14,GOOGL,Purchase,"$1,001 - $15,000",Representatives,Democrat,Stock


## 2. Normalisation → schéma cible du projet

Chaque règle est tracée : chambre déduite, dates en ISO, opération normalisée, montant → midpoint
(`None` si tranche hors barème, à documenter), `non trouvé` explicite plutôt qu'une valeur inventée.

In [3]:
MIDPOINTS = {
    "$1,001 - $15,000": 8000.5, "$15,001 - $50,000": 32500.5,
    "$50,001 - $100,000": 75000.5, "$100,001 - $250,000": 175000.5,
    "$250,001 - $500,000": 375000.5, "$500,001 - $1,000,000": 750000.5,
    "$1,000,001 - $5,000,000": 3000000.5, "$5,000,001 - $25,000,000": 15000000.5,
    "$25,000,001 - $50,000,000": 37500000.5,
}
NA = "non trouve"

def norm_chamber(v):
    v = (v or "").lower()
    if "senate" in v:                          return "Senate"   # 'representatives' contient 'sen' -> on exige 'senate'
    if "house" in v or "representative" in v:  return "House"
    return NA

def norm_op(v):
    v = (v or "").strip().lower()
    if v.startswith("purchase"): return "Purchase"
    if "partial" in v:           return "Partial Sale"
    if "sale" in v:              return "Sale"
    if "exchange" in v:          return "Exchange"
    return v or NA

def norm_date(v):
    v = (v or "").strip()
    for fmt in ("%Y-%m-%d", "%m/%d/%Y"):
        try: return datetime.strptime(v, fmt).date().isoformat()
        except ValueError: pass
    return None

def range_midpoint(v):
    return MIDPOINTS.get((v or "").strip())   # None si tranche inconnue -> a documenter

def normalize_quiver(raw):
    rows = []
    for d in raw:
        rng = (d.get("Range") or "").strip()
        rows.append({
            "declarant_name":  d.get("Representative") or NA,
            "chamber":         norm_chamber(d.get("House")),
            "party":           d.get("Party") or NA,
            "transaction_date": norm_date(d.get("TransactionDate")),
            "disclosure_date":  norm_date(d.get("ReportDate")),
            "ticker":          (d.get("Ticker") or NA).upper(),
            "operation_type":  norm_op(d.get("Transaction")),
            "amount_range":    rng or NA,
            "amount_midpoint": range_midpoint(rng),
            "asset_type":      d.get("TickerType") or NA,
            "source":          "quiver",
        })
    return pd.DataFrame(rows)

qv = normalize_quiver(raw_quiver)
print(f"{len(qv)} records normalises.")
display(qv.head())

12 records normalises.


,declarant_name,chamber,party,transaction_date,disclosure_date,ticker,operation_type,amount_range,amount_midpoint,asset_type,source
0,Mark Alford,House,Republican,2024-03-16,2024-03-31,NVDA,Purchase,"$1,001 - $15,000",8000.5,Stock,quiver
1,Mark Alford,House,Republican,2024-03-18,2024-04-02,MSFT,Sale,"$15,001 - $50,000",32500.5,Stock,quiver
2,Marjorie Taylor Greene,House,Republican,2024-04-22,2024-05-10,AAPL,Purchase,"$1,001 - $15,000",8000.5,Stock,quiver
3,Josh Gottheimer,House,Democrat,2024-05-20,2024-06-05,AMZN,Purchase,"$15,001 - $50,000",32500.5,Stock,quiver
4,Josh Gottheimer,House,Democrat,2024-06-14,2024-07-01,GOOGL,Purchase,"$1,001 - $15,000",8000.5,Stock,quiver


## 3. Audit de couverture (le plafond ~1 800 se MESURE, il ne se recopie pas)

In [4]:
def year_of(row):  # annee a partir de la transaction_date (fallback disclosure)
    return (row["transaction_date"] or row["disclosure_date"] or "")[:4]

qv["year"] = qv.apply(year_of, axis=1)
yrs = qv["year"].replace("", pd.NA).dropna()

cov = {
    "source": "quiver",
    "premiere_annee": yrs.min() if len(yrs) else NA,
    "derniere_annee": yrs.max() if len(yrs) else NA,
    "n_records": len(qv),
    "n_tickers_distincts": int(qv.loc[qv["ticker"] != NA, "ticker"].nunique()),
    "n_declarants": int(qv["declarant_name"].nunique()),
    "n_house": int((qv["chamber"] == "House").sum()),
    "n_senate": int((qv["chamber"] == "Senate").sum()),
}
coverage_summary = pd.DataFrame([cov])
coverage_summary.to_csv(OUT_DIR / "coverage_summary.csv", index=False)

print(f"Tickers distincts MESURES : {cov['n_tickers_distincts']}  "
      f"(annonce par Quiver : ~1 800 -> a confronter sur un vrai pull)")
display(coverage_summary)
print("Volume par annee :");        display(qv["year"].value_counts().sort_index())
print("Repartition par chambre :"); display(qv["chamber"].value_counts())
print("Repartition par parti :");   display(qv["party"].value_counts())

Tickers distincts MESURES : 10  (annonce par Quiver : ~1 800 -> a confronter sur un vrai pull)


,source,premiere_annee,derniere_annee,n_records,n_tickers_distincts,n_declarants,n_house,n_senate
0,quiver,2023,2024,12,10,6,9,3


Volume par annee :


year
2023     2
2024    10
Name: count, dtype: int64

Repartition par chambre :


chamber
House     9
Senate    3
Name: count, dtype: int64

Repartition par parti :


party
Republican    9
Democrat      3
Name: count, dtype: int64

## 4. Audit des champs — complétude et cohérence des **deux dates**

In [5]:
def missing_mask(col):
    return qv[col].isna() | (qv[col].astype(str).isin([NA, "", "None"]))

field_rows = []
for c in ["declarant_name","chamber","party","transaction_date","disclosure_date",
          "ticker","operation_type","amount_range","amount_midpoint","asset_type"]:
    m = missing_mask(c)
    field_rows.append({"champ": c, "n_manquant": int(m.sum()),
                       "taux_manquant_%": round(100*m.mean(), 1)})
field_completeness = pd.DataFrame(field_rows)
field_completeness.to_csv(OUT_DIR / "field_completeness.csv", index=False)
display(field_completeness)

both = qv["transaction_date"].notna() & qv["disclosure_date"].notna()
print(f"Les deux dates presentes : {both.mean()*100:.0f}% des records")
if both.any():
    gap = (pd.to_datetime(qv.loc[both,"disclosure_date"]) -
           pd.to_datetime(qv.loc[both,"transaction_date"])).dt.days
    print(f"disclosure >= transaction : {(gap >= 0).mean()*100:.0f}% des records a deux dates")
    print(f"Delai disclosure-transaction (jours) : mediane {gap.median():.0f}, max {gap.max():.0f}  "
          f"(rappel : PTR <= ~45 j ; au-dela = signal en retard)")

,champ,n_manquant,taux_manquant_%
0,declarant_name,0,0.0
1,chamber,0,0.0
2,party,0,0.0
3,transaction_date,0,0.0
4,disclosure_date,0,0.0
5,ticker,0,0.0
6,operation_type,0,0.0
7,amount_range,0,0.0
8,amount_midpoint,0,0.0
9,asset_type,0,0.0


Les deux dates presentes : 100% des records
disclosure >= transaction : 100% des records a deux dates
Delai disclosure-transaction (jours) : mediane 18, max 26  (rappel : PTR <= ~45 j ; au-dela = signal en retard)


## 5. Validation croisée vs extraction officielle House — **le test décisif**

Sur `OVERLAP_YEAR`, on apparie le sous-ensemble **House** de Quiver à ton extraction officielle sur
`(déclarant, ticker, transaction_date, fourchette)`, puis on classe les écarts.

In [6]:
def _ensure_house_fixture(path: Path):
    """Extraction officielle House SYNTHETIQUE pour OVERLAP_YEAR si absente.
    /!\\ DONNEES DE DEMO. En vrai : pointe HOUSE_EXTRACT_PATH sur la sortie de ton pipeline."""
    if path.exists():
        return
    demo = [
        {"declarant_name":"Hon. Mark Alford (MO04)","chamber":"House","transaction_date":"2024-03-16","ticker":"NVDA","operation_type":"Purchase","amount_range":"$1,001 - $15,000"},
        {"declarant_name":"Hon. Mark Alford (MO04)","chamber":"House","transaction_date":"2024-03-18","ticker":"MSFT","operation_type":"Sale","amount_range":"$15,001 - $50,000"},
        {"declarant_name":"Hon. Marjorie Taylor Greene (GA14)","chamber":"House","transaction_date":"2024-04-22","ticker":"AAPL","operation_type":"Purchase","amount_range":"$1,001 - $15,000"},
        {"declarant_name":"Hon. Josh Gottheimer (NJ05)","chamber":"House","transaction_date":"2024-05-20","ticker":"AMZN","operation_type":"Purchase","amount_range":"$15,001 - $50,000"},
        {"declarant_name":"Hon. Josh Gottheimer (NJ05)","chamber":"House","transaction_date":"2024-06-14","ticker":"GOOGL","operation_type":"Purchase","amount_range":"$1,001 - $15,000"},
        {"declarant_name":"Hon. Mark Alford (MO04)","chamber":"House","transaction_date":"2024-07-09","ticker":"TSLA","operation_type":"Purchase","amount_range":"$1,001 - $15,000"},
        {"declarant_name":"Hon. Marjorie Taylor Greene (GA14)","chamber":"House","transaction_date":"2024-08-15","ticker":"AMD","operation_type":"Purchase","amount_range":"$15,001 - $50,000"},
        {"declarant_name":"Hon. Daniel Meuser (PA09)","chamber":"House","transaction_date":"2024-09-30","ticker":"COST","operation_type":"Purchase","amount_range":"$1,001 - $15,000"},
    ]
    pd.DataFrame(demo).to_csv(path, index=False)
    log.warning("Fixture House officielle de demonstration ecrite : %s (DONNEES NON REELLES)", path)

def name_key(s):
    s = re.sub(r"\(.*?\)", "", s or "")                              # enleve (MO04)
    s = re.sub(r"\b(hon|mr|mrs|ms|dr|sen|rep)\.?\b", "", s, flags=re.I)
    toks = re.sub(r"[^a-zA-Z ]", " ", s).split()
    return " ".join(sorted(t.upper() for t in toks))                 # insensible a l'ordre prenom/nom

_ensure_house_fixture(HOUSE_EXTRACT_PATH)
off = pd.read_csv(HOUSE_EXTRACT_PATH)
off["key"] = off["declarant_name"].map(name_key)

agg_h = qv[(qv["chamber"] == "House") & (qv["year"] == str(OVERLAP_YEAR))].copy()
agg_h["key"] = agg_h["declarant_name"].map(name_key)
print(f"Quiver House {OVERLAP_YEAR} : {len(agg_h)} records  |  officiel : {len(off)} records")

keys = ["key","ticker","transaction_date","amount_range"]
m = agg_h.merge(off, on=keys, how="outer", indicator=True, suffixes=("_qv","_off"))
matched  = int((m["_merge"] == "both").sum())
only_agg = int((m["_merge"] == "left_only").sum())
only_off = int((m["_merge"] == "right_only").sum())

la = m[m["_merge"] == "left_only"][["key","ticker","transaction_date","amount_range"]]
ro = m[m["_merge"] == "right_only"][["key","ticker","transaction_date","amount_range"]]
date_mismatch = int(la.merge(ro, on=["key","ticker","amount_range"]).shape[0])      # meme range, date differente
amt_mismatch  = int(la.merge(ro, on=["key","ticker","transaction_date"]).shape[0])  # meme date, montant different
only_agg_pur = only_agg - date_mismatch - amt_mismatch
only_off_pur = only_off - date_mismatch - amt_mismatch
union = matched + date_mismatch + amt_mismatch + only_agg_pur + only_off_pur

m.to_csv(OUT_DIR / "crossval_discrepancies.csv", index=False)
crossval = pd.DataFrame([{
    "overlap_year": OVERLAP_YEAR, "apparies_exacts": matched,
    "dates_divergentes": date_mismatch, "montants_divergents": amt_mismatch,
    "seulement_quiver": only_agg_pur, "seulement_officiel": only_off_pur,
    "taux_accord_%": round(100*matched/union, 1) if union else 0.0,
}])
display(crossval)
print("Detail ligne a ligne -> crossval_discrepancies.csv")

WARNING Fixture House officielle de demonstration ecrite : out_aggregators_audit/house_official_2024.csv (DONNEES NON REELLES)


Quiver House 2024 : 8 records  |  officiel : 8 records


,overlap_year,apparies_exacts,dates_divergentes,montants_divergents,seulement_quiver,seulement_officiel,taux_accord_%
0,2024,5,1,1,1,1,55.6


Detail ligne a ligne -> crossval_discrepancies.csv


## 6. Contrôle d'intégrité sur échantillon (vérif manuelle)

Leçon de l'incident JSON de la slide : **ne jamais faire confiance à un label « réel » sans recoupement.**
On tire quelques records et on les confronte à Capitol Trades / au filing source — en vérifiant **les deux dates**.

In [7]:
ech = qv.sample(min(6, len(qv)), random_state=0)[
    ["declarant_name","chamber","ticker","operation_type","transaction_date","disclosure_date","amount_range"]
].reset_index(drop=True)
ech["a_verifier_sur"] = "Capitol Trades / filing source -> rechercher : " + ech["declarant_name"]

print("A VERIFIER A LA MAIN (confirmer ticker + LES DEUX DATES + montant) :")
print("Capitol Trades (gratuit) : https://www.capitoltrades.com  -> rechercher le declarant")
display(ech)

A VERIFIER A LA MAIN (confirmer ticker + LES DEUX DATES + montant) :
Capitol Trades (gratuit) : https://www.capitoltrades.com  -> rechercher le declarant


,declarant_name,chamber,ticker,operation_type,transaction_date,disclosure_date,amount_range,a_verifier_sur
0,Marjorie Taylor Greene,House,AMD,Purchase,2024-08-15,2024-09-02,"$50,001 - $100,000",Capitol Trades / filing source -> rechercher :...
1,Tommy Tuberville,Senate,AAPL,Purchase,2023-07-29,2023-08-20,"$1,001 - $15,000",Capitol Trades / filing source -> rechercher :...
2,Josh Gottheimer,House,GOOGL,Purchase,2024-06-14,2024-07-01,"$1,001 - $15,000",Capitol Trades / filing source -> rechercher :...
3,Josh Gottheimer,House,DIS,Sale,2023-02-20,2023-03-15,"$1,001 - $15,000",Capitol Trades / filing source -> rechercher :...
4,Marjorie Taylor Greene,House,AAPL,Purchase,2024-04-22,2024-05-10,"$1,001 - $15,000",Capitol Trades / filing source -> rechercher :...
5,John Boozman,Senate,NVDA,Purchase,2024-03-19,2024-04-14,"$1,001 - $15,000",Capitol Trades / filing source -> rechercher :...


## 7. Verdict & manifeste

In [8]:
accord = float(crossval["taux_accord_%"].iloc[0])
verdict = pd.DataFrame([
    {"question": "Fallback recent (>= 2016) ?",
     "reponse": f"Plausible si l'accord est eleve - mesure ici : {accord}% sur {OVERLAP_YEAR}"},
    {"question": "Fallback historique Senat 2013-2015 ?",
     "reponse": "NON - Quiver demarre en janv. 2016, il ne peut pas combler ce trou."},
    {"question": "Fallback Senat 2016-2019 ?",
     "reponse": "Depend de la couverture Senat mesuree (section 3) - a trancher sur un vrai pull."},
    {"question": "Capitol Trades ?",
     "reponse": "Verification visuelle/manuelle seulement (pas d'API publique, ~3 ans d'historique)."},
])
display(verdict)

manifest = pd.DataFrame([{
    "source": "quiver", "mode": "OFFLINE (demo)" if OFFLINE else "LIVE",
    "ok": len(qv) > 0, "n_records": len(qv),
    "annees": f"{cov['premiere_annee']}-{cov['derniere_annee']}",
    "tickers_distincts_mesures": cov["n_tickers_distincts"],
    "taux_accord_overlap_%": accord,
}])
manifest.to_csv(OUT_DIR / "manifest.csv", index=False)
display(manifest)
print()
print("Sorties ecrites dans", OUT_DIR, ":")
for p in sorted(OUT_DIR.glob("*.csv")): print("  -", p.name)

,question,reponse
0,Fallback recent (>= 2016) ?,Plausible si l'accord est eleve - mesure ici :...
1,Fallback historique Senat 2013-2015 ?,"NON - Quiver demarre en janv. 2016, il ne peut..."
2,Fallback Senat 2016-2019 ?,Depend de la couverture Senat mesuree (section...
3,Capitol Trades ?,Verification visuelle/manuelle seulement (pas ...


,source,mode,ok,n_records,annees,tickers_distincts_mesures,taux_accord_overlap_%
0,quiver,OFFLINE (demo),True,12,2023-2024,10,55.6



Sorties ecrites dans out_aggregators_audit :
  - coverage_summary.csv
  - crossval_discrepancies.csv
  - field_completeness.csv
  - house_official_2024.csv
  - manifest.csv


## 8. À reporter dans l'audit — et ce que tu me colles

**Dans le deck (slide agrégateurs)** :
- la fenêtre temporelle **mesurée** (≥ 2016) et le **nombre de tickers distincts mesuré** (vs ~1 800 annoncé) ;
- le **taux d'accord** sur l'année de recouvrement + le décompte des écarts (dates / montants / présence) ;
- le verdict : Quiver = validation croisée / fallback récent ; **inutile pour le trou Sénat pré-2016** ;
  Capitol Trades = vérification manuelle.

**Ce que tu me colles** (pour que je finalise) :
1. Le `coverage_summary.csv` (surtout `n_tickers_distincts` réel et la fenêtre d'années).
2. La ligne de `crossval` (taux d'accord + écarts) **après** un vrai pull avec ta clé sur `OVERLAP_YEAR`.
3. 2–3 records vérifiés à la main sur Capitol Trades (les deux dates confirmées).

> **Important** : en mode OFFLINE, tous les chiffres ci-dessus portent sur l'**échantillon de démonstration synthétique**,
> pas sur des données réelles. Mets `QUIVER_API_KEY` (et pointe `HOUSE_EXTRACT_PATH` sur ta vraie extraction)
> pour produire des chiffres réels.